Cleaning of customers data
=======================================

1. create a data rame with a proper datatypes
2.. Rename a few columns
annual_inc-> annual_income
addr_state-> address_state
zip_cod-> address_zipcode
country-> address_country
tot_hi_cred_lim-> total_high_credit_limit
annual_inc_joint-> join_annual_income
3. insert a new column named as ingestion date (current time)
4. remove complete duplicate rows
5. remove the rows where annual_income is null
6. convert emp_length to integer
7. we need to replace all the nulls in emp_length column with average of this
column
8. clean the address_state (it should be 2 characters only), replace all others
with NA

In [133]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()
spark =SparkSession. \
        builder. \
        config('spark.ui.port','0'). \
        config('spark.shuffle.useOldFetchProtocol','true'). \
        config('spark.sql.warehouse.dir',f"/user/{username}/warehouse"). \
        enableHiveSupport(). \
        master('yarn'). \
        getOrCreate()

In [134]:
customers_schema = 'member_id string, emp_title string, empl_length string, home_ownership string, annual_inc float,addr_state string, zip_code string, country string, grade string, sub_grade string, verification_status string, tot_hi_cred_lim  float,application_type string,annual_inc_joint float,verfication_status_joint string' 

In [135]:
customers_raw_df = spark.read \
.format("csv") \
.schema(customers_schema) \
.load("/user/itv026162/lendingclub/project/raw/customers_data_csv")

In [136]:
customers_raw_df

member_id,emp_title,empl_length,home_ownership,annual_inc,addr_state,zip_code,country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,annual_inc_joint,verfication_status_joint
member_id,emp_title,emp_length,home_ownership,null,addr_state,zip_code,country,grade,sub_grade,verification_status,null,application_type,null,verification_stat...
961ae110ec063761f...,Crothall Services...,3 years,RENT,28685.0,NY,104xx,USA,C,C1,Not Verified,null,Individual,null,null
291854d244748c911...,PharMerica,1 year,RENT,111000.0,TX,757xx,USA,B,B3,Not Verified,null,Individual,null,null
b5cf66b90193da9d2...,Robert Half Inter...,3 years,RENT,38000.0,CA,956xx,USA,A,A5,Not Verified,null,Individual,null,null
7f53fd1acb7759e5b...,walmart,2 years,RENT,17000.0,NY,130xx,USA,D,D1,Not Verified,null,Individual,null,null
12edc8e328b993914...,Delloite,10+ years,MORTGAGE,500000.0,GA,300xx,USA,C,C4,Not Verified,null,Individual,null,null
e18a847f091332ff6...,Hightower Investm...,3 years,OWN,65000.0,NY,100xx,USA,B,B5,Not Verified,null,Individual,null,null
d6fe0000e876ab76d...,double barrel env...,2 years,MORTGAGE,48000.0,CA,933xx,USA,B,B5,Not Verified,null,Individual,null,null
62c38934f3a2ebcd2...,e-Dialog,3 years,MORTGAGE,68500.0,MA,021xx,USA,D,D4,Not Verified,null,Individual,null,null
7701762ef5d302b0d...,City of Hialeah,10+ years,MORTGAGE,80000.0,FL,331xx,USA,D,D3,Not Verified,null,Individual,null,null


In [137]:
customers_raw_df.printSchema()

#annual income cannot be string, it should be float, cause its money
#total credit limit shouwld also be float
#annual income joint should also be float

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- empl_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: float (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- tot_hi_cred_lim: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- annual_inc_joint: float (nullable = true)
 |-- verfication_status_joint: string (nullable = true)



In [138]:
customers_df_renamed = customers_raw_df.withColumnRenamed("annual_inc","annual_income") \
.withColumnRenamed("addr_state","address_state") \
.withColumnRenamed("zip_code","address_zipcode") \
.withColumnRenamed("country","address_country") \
.withColumnRenamed("tot_hi_cred_lim","total_high_credit_limit") \
.withColumnRenamed("annual_inc_joint","join_annual_income")

#changing columns names for easy readability

In [139]:
from pyspark.sql.functions import current_timestamp

In [140]:
customers_df_ingested = customers_df_renamed.withColumn("ingest_date",current_timestamp())

In [141]:
customers_df_ingested

member_id,emp_title,empl_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verfication_status_joint,ingest_date
member_id,emp_title,emp_length,home_ownership,null,addr_state,zip_code,country,grade,sub_grade,verification_status,null,application_type,null,verification_stat...,2026-11-17 19:53:...
961ae110ec063761f...,Crothall Services...,3 years,RENT,28685.0,NY,104xx,USA,C,C1,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
291854d244748c911...,PharMerica,1 year,RENT,111000.0,TX,757xx,USA,B,B3,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
b5cf66b90193da9d2...,Robert Half Inter...,3 years,RENT,38000.0,CA,956xx,USA,A,A5,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
7f53fd1acb7759e5b...,walmart,2 years,RENT,17000.0,NY,130xx,USA,D,D1,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
12edc8e328b993914...,Delloite,10+ years,MORTGAGE,500000.0,GA,300xx,USA,C,C4,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
e18a847f091332ff6...,Hightower Investm...,3 years,OWN,65000.0,NY,100xx,USA,B,B5,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
d6fe0000e876ab76d...,double barrel env...,2 years,MORTGAGE,48000.0,CA,933xx,USA,B,B5,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
62c38934f3a2ebcd2...,e-Dialog,3 years,MORTGAGE,68500.0,MA,021xx,USA,D,D4,Not Verified,null,Individual,null,null,2026-11-17 19:53:...
7701762ef5d302b0d...,City of Hialeah,10+ years,MORTGAGE,80000.0,FL,331xx,USA,D,D3,Not Verified,null,Individual,null,null,2026-11-17 19:53:...


In [142]:
customers_df_ingested.count()

2260702

In [143]:
customers_distinct = customers_df_ingested.distinct() #distinct is a wide transformation it will take a bit of time
#wide trasnforamtions - meaning the shuffling of data happens i,e. actions happen locally but for global output it needs to go one machine and then provides the output.

In [144]:
customers_distinct.count()

2260639

In [145]:
2260702-2260639 #number of duplicates present

63

Just creating a temporary table for few checkpoints
======================

In [146]:
customers_distinct.createOrReplaceTempView("customers")

In [147]:
spark.sql("select * from customers limit 5")

member_id,emp_title,empl_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verfication_status_joint,ingest_date
27a626c10cae9ca36...,owner,< 1 year,RENT,26200.0,NY,112xx,USA,B,B5,Source Verified,76833.0,Individual,null,null,2026-11-17 19:53:...
d20f6f13c3ec4c87b...,Sr SQL Developer,2 years,MORTGAGE,210000.0,GA,300xx,USA,A,A4,Source Verified,223444.0,Individual,null,null,2026-11-17 19:53:...
0be5e2ed96586de92...,Machinist,3 years,MORTGAGE,50000.0,MN,554xx,USA,B,B4,Source Verified,156300.0,Individual,null,null,2026-11-17 19:53:...
855cec661f392e370...,Dental Assistant,10+ years,MORTGAGE,12000.0,LA,707xx,USA,B,B5,Not Verified,74758.0,Joint App,74400.0,Not Verified,2026-11-17 19:53:...
33158394404935655...,Latent Fingerprin...,3 years,MORTGAGE,125000.0,FL,346xx,USA,D,D2,Verified,350724.0,Individual,null,null,2026-11-17 19:53:...


In [148]:
spark.sql("select count(*) from customers where annual_income is null")

count(1)
6


In [149]:
customers_income_filtered = spark.sql("""
select * 
from customers
where annual_income is not null
""")

In [150]:
customers_income_filtered.createOrReplaceTempView("customers")

In [151]:
spark.sql("select distinct(empl_length) from customers").show()

+-----------+
|empl_length|
+-----------+
|    9 years|
|    5 years|
|       null|
|     1 year|
|    2 years|
|    7 years|
|    8 years|
|    4 years|
|    6 years|
|    3 years|
|  10+ years|
|   < 1 year|
+-----------+



In [152]:
from pyspark.sql.functions import regexp_replace, col

In [153]:
customers_emplength_cleaned = customers_income_filtered.withColumn("empl_length", regexp_replace(col("empl_length"),"(\D)",""))

#so we are using regexp_replace to remove the non digit to a empty, in the same column.
#\D means not a digit
#for that we have to import the respective functions.

In [154]:
customers_emplength_cleaned

member_id,emp_title,empl_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verfication_status_joint,ingest_date
a063c15700369289b...,null,null,MORTGAGE,48826.0,CA,958xx,USA,B,B1,Verified,261715.0,Individual,null,null,2026-11-17 19:54:...
1f083b8c4ff57d4c5...,null,1,RENT,50000.0,SC,292xx,USA,B,B1,Not Verified,44000.0,Individual,null,null,2026-11-17 19:54:...
e7765d98ca34d049a...,Baker,10,MORTGAGE,42000.0,NH,035xx,USA,A,A2,Not Verified,124592.0,Individual,null,null,2026-11-17 19:54:...
54d3d94a79d050020...,US COURTS,10,MORTGAGE,78000.0,TX,785xx,USA,A,A2,Verified,506569.0,Joint App,198000.0,null,2026-11-17 19:54:...
6d6062e3e40af90f2...,null,null,RENT,27860.0,KY,403xx,USA,C,C3,Source Verified,23598.0,Individual,null,null,2026-11-17 19:54:...
0a6019e6d6284f926...,Relationship Manager,7,RENT,45000.0,CA,913xx,USA,C,C1,Verified,59643.0,Individual,null,null,2026-11-17 19:54:...
cb0e81698cb2c05bf...,null,null,RENT,15312.0,AR,720xx,USA,C,C4,Verified,16900.0,Individual,null,null,2026-11-17 19:54:...
7428e16afc1034d6e...,Laundry Attendant,1,RENT,25000.0,TN,378xx,USA,C,C1,Source Verified,44100.0,Joint App,50000.0,null,2026-11-17 19:54:...
069a8d5481b8c50e3...,Kennel Technican,1,OWN,23000.0,IN,461xx,USA,C,C5,Source Verified,11468.0,Individual,null,null,2026-11-17 19:54:...
3053bee506506f9e7...,Industrial Engine...,10,MORTGAGE,120000.0,ME,040xx,USA,A,A2,Source Verified,404622.0,Individual,null,null,2026-11-17 19:54:...


In [155]:
customers_emplength_cleaned.printSchema()
#if you observe we have empl_length as string, which doesnt make any sense, so next line we are going to cast into "int"

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- empl_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_income: float (nullable = true)
 |-- address_state: string (nullable = true)
 |-- address_zipcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- total_high_credit_limit: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- join_annual_income: float (nullable = true)
 |-- verfication_status_joint: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [156]:
customers_emplength_casted = customers_emplength_cleaned.withColumn("empl_length",customers_emplength_cleaned.empl_length.cast('int'))

In [157]:
customers_emplength_casted.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- empl_length: integer (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_income: float (nullable = true)
 |-- address_state: string (nullable = true)
 |-- address_zipcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- total_high_credit_limit: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- join_annual_income: float (nullable = true)
 |-- verfication_status_joint: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [158]:
customers_emplength_casted.filter("empl_length is null").count()

146903

we cannot remove these many customers, but as per the stakeholder when I have reached out with the insight, they have suggested to not to remove the customers, but replace the null values by the avg value of empl_length

In [159]:
customers_emplength_casted.createOrReplaceTempView("customers")

In [160]:
spark.sql("select avg(empl_length) as average_value from customers")

average_value
6.021258628112389


Once again checked with the manager on this as the avg value is a float, but want to keep it to an int i,e round off 
hence going to use floor function

In [161]:
spark.sql("select floor(avg(empl_length)) as average_value from customers")

average_value
6


In [162]:
avg_emp_length = spark.sql("select floor(avg(empl_length)) as average_value from customers").collect()

In [163]:
avg_emp_duration = avg_emp_length[0][0]

In [164]:
print(avg_emp_duration)

6


In [165]:
customer_emplength_replaced = customers_emplength_casted.na.fill(avg_emp_duration,subset=['empl_length'])

In [166]:
customer_emplength_replaced.filter("empl_length is null").count()

0

So we have updated the empl_length which are null to 6, hence when we filter the data, we get 0

In [167]:
spark.sql("select distinct(address_state) from customers")

address_state
Helping Kenya's D...
223xx
175 (total projec...
SC
AZ
"so Plan """"C"""" is ..."
I am 56 yrs. old ...
financially I mad...
but no one will l...
LA


Okay, so if I see the Address_state, right? So what we can observe is like LA,MN , AZ, SC. These are the valid ones. But however, we can see there is a lot of messed up data. So what I'm gonna do right now is I'm gonna check how much data is messed up, and if it's a lot— it's a lot—and check with my stakeholder, like what exactly they are getting the data, and, like, get the sourcing right, or to understand the concerns, and then we can take it forward. But however, let me check how it goes now.

In [168]:
spark.sql("select count(address_state) from customers where length(address_state) > 2")

count(address_state)
254


In [169]:
from pyspark.sql.functions import when, col, length

In [170]:
customers_state_cleaned = customer_emplength_replaced.withColumn(
"address_state",
    when(length(col("address_state")) > 2, "NA").otherwise(col("address_state"))
)

In [171]:
customers_state_cleaned.select("address_state").distinct()

address_state
AZ
SC
LA
MN
NJ
DC
OR
NA
VA
null


Standardize address state values: values longer than three characters are
replaced with "NA"; all other values are kept as provided. Null values remain
null because they are missing values. The customer team can decide whether
further analysis or customer follow-up is needed.

In [172]:
customers_state_cleaned

member_id,emp_title,empl_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verfication_status_joint,ingest_date
c226d082b3e8654a1...,Finisher,10,MORTGAGE,60000.0,AL,357xx,USA,B,B1,Not Verified,319539.0,Individual,null,null,2026-11-17 19:55:...
b72dcdca942afd65b...,Truck driver,9,MORTGAGE,52000.0,AL,363xx,USA,C,C2,Verified,72930.0,Individual,null,null,2026-11-17 19:55:...
4c701ef2bda629204...,Financial Coordin...,2,MORTGAGE,60000.0,CT,064xx,USA,C,C4,Not Verified,301015.0,Individual,null,null,2026-11-17 19:55:...
b0b2ac0cda6138817...,Sr. Project Engineer,10,OWN,150000.0,WI,530xx,USA,B,B3,Not Verified,772667.0,Individual,null,null,2026-11-17 19:55:...
c7874b48034186261...,Pharmacy Manager,10,OWN,143000.0,AZ,856xx,USA,D,D2,Verified,544781.0,Individual,null,null,2026-11-17 19:55:...
1f6d7bea70de25a04...,cdl driver,5,OWN,60000.0,NV,890xx,USA,C,C1,Not Verified,26245.0,Individual,null,null,2026-11-17 19:55:...
b1243168c17b5e991...,Supply Chain Lead,3,MORTGAGE,80000.0,PA,179xx,USA,C,C2,Not Verified,278953.0,Individual,null,null,2026-11-17 19:55:...
9c858c2fe581234ab...,Service Manager,1,MORTGAGE,92000.0,WA,983xx,USA,D,D3,Source Verified,369974.0,Joint App,137000.0,Not Verified,2026-11-17 19:55:...
981be56ad0a90e6e1...,Assistant Propert...,1,MORTGAGE,46000.0,NV,895xx,USA,D,D2,Source Verified,153998.0,Individual,null,null,2026-11-17 19:55:...
b8fc87d331a9829b7...,Finance Manager,1,MORTGAGE,40000.0,MI,486xx,USA,D,D2,Verified,269091.0,Joint App,104000.0,Not Verified,2026-11-17 19:55:...


In [173]:
customers_state_cleaned.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv026162/lendingclub/project/cleaned/customers_parquet") \
.save()

As per the stakeholder's request, we are saving the data in the Parquet format for the ease of the calculation and to keep the data storage lesser. That's why we are using it in the user rather than keeping it on the public folder. So we are using in the Parquet form.

In [174]:
customers_state_cleaned.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned/customers_csv") \
.save()

In [175]:
spark.stop()